# 06 — Locked Predictive Distributions

This notebook fits the locked probabilistic specification using all
warm-up and development observations.

It then generates holdout and June predictive distributions using the
dispersion scale selected in Notebook 05.

No realised holdout or June outcome enters model fitting or appears in
the prediction output.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "config/"
            "locked_prediction_spec.yaml"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/"
            "generate_locked_evaluation_predictions.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "Locked prediction generation failed."
    )


 LOCKED HOLDOUT AND EXTERNAL PREDICTIONS GENERATED

Selected model: pooled_empirical_residual
Selected family: empirical_residual
Locked dispersion scale: 1.25

Training rows: 216
Training dates: 62
Training period: 2026-03-16 to 2026-05-21

Prediction rows: 159
Prediction dates: 40
Holdout rows: 40
External-test rows: 119

Maximum predictive median change: 0.0
Positive central 80% width share: 1.0
Final fit warnings: 0

Prediction block summary:
chronology_block  rows  dates start_date   end_date  decision_rules
         holdout    40     10 2026-05-22 2026-05-31               4
   external_test   119     30 2026-06-01 2026-06-30               4

Holdout outcomes used for fit: False
External outcomes used for fit: False
Continuous scores calculated: False
Event probabilities calculated: False
Market data accessed: False
Trading returns calculated: False

Next stage: join realised outcomes only after these prediction files are locked, then evaluate holdout and external continuous-dist

## Final fitting sample

The selected model is fitted using only the warm-up and development
blocks. The fitting sample therefore ends before the first holdout
date.

No refitting occurs after observing holdout outcomes.

In [2]:
manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "06_locked_prediction_manifest.json"
    ).read_text(encoding="utf-8")
)

blocks = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "06_locked_prediction_block_summary.csv"
)

print("Selected model:", manifest["selected_model"])
print("Selected family:", manifest["selected_family"])
print(
    "Dispersion scale:",
    manifest["selected_dispersion_scale"],
)
print()
print(
    "Training period:",
    manifest["training_start"],
    "to",
    manifest["training_end"],
)
print()
print(blocks.to_string(index=False))

Selected model: pooled_empirical_residual
Selected family: empirical_residual
Dispersion scale: 1.25

Training period: 2026-03-16 to 2026-05-21

chronology_block  rows  dates start_date   end_date  decision_rules
         holdout    40     10 2026-05-22 2026-05-31               4
   external_test   119     30 2026-06-01 2026-06-30               4


## Locked prediction files

Both the uncalibrated and calibrated distributions contain 99
quantiles.

Calibration changes the dispersion around the median but leaves the
predictive median unchanged.

In [3]:
calibrated = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "06_locked_calibrated_predictions.csv"
)

prohibited = [
    column
    for column in calibrated.columns
    if any(
        term in column.lower()
        for term in [
            "hko_daily_max",
            "residual",
            "outcome",
            "crps",
            "brier",
            "pnl",
            "market_price",
        ]
    )
]

assert not prohibited
assert len(calibrated) == 159
assert calibrated["target_date"].nunique() == 40

print(
    "Locked prediction rows:",
    len(calibrated),
)

print(
    "Locked prediction dates:",
    calibrated["target_date"].nunique(),
)

Locked prediction rows: 159
Locked prediction dates: 40


## Evidential boundary

This notebook generates forecasts but does not evaluate them.

Continuous scoring, event probability construction, market comparison
and trading analysis are separate later stages.

In [4]:
assert manifest["predictions_locked"] is True
assert manifest["holdout_outcomes_used_for_fit"] is False
assert manifest["external_test_outcomes_used_for_fit"] is False
assert manifest["continuous_scores_calculated"] is False
assert manifest["event_probabilities_calculated"] is False
assert manifest["market_data_accessed"] is False
assert manifest["trading_returns_calculated"] is False

print(
    "Predictions locked without evaluation:",
    True,
)

Predictions locked without evaluation: True
